In [20]:
import pandas as pd
import os
import re
import csv

def validate_precision(value):
    try:
        pd.to_datetime(value)
        return 1
    except:
        return 0

def validate_consistency(value):
    pattern = '^[a-zA-Z0-9áéíóúüÀÁÉÍÓÚÜñÑçã&_. -]*$'
    return 1 if re.match(pattern, str(value)) else 0

def validate_validity(value):
    return 0 if pd.isna(value) else 1

def validate_integrity(value):
    if isinstance(value, (int, float)):
        return 0 if value < 0 else 1
    elif isinstance(value, str):
        if '.' in value or ' ' in value or '-' in value or 'Sin información' in value or '-1' in value or 'No definido' in value or 'N/A' in value or 'Sin Clasificar':
            return 0
    return 1


# Asignación de variables
file_path = 'Entrada_Fuentes_de_Datos/Extraccion_Analisis_Datos_Faculty360.xlsx'
columnas_unicidad = ['Nomina','Contrato']
columnas_precision = ['Fecha Inicio Contrato','Fecha Fin Contrato','Fecha_Acreditacion','fecha_ultima_modificacion','DateAccreditation','CreationDate']
columnas_integridad = ['Antigüedad']  

print(f'Inicio del proceso, espere unos minutos..\n\n')
with pd.ExcelFile(file_path) as excel:
    for sheet_name in excel.sheet_names:
        if  sheet_name not in('DWH - Success Factor','Profesor Escuela Top','Profesor Facultad','Profesor Grado Académico'):
            continue
            
        print(f'Pestaña analizada: {sheet_name}')    
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        output_data = []
        
        for i, row in df.iterrows():
            nomina = row.iloc[0] #primera columna siempre es nómina
            for col in df.columns[1:]:
                value = row[col]
                if col in columnas_precision:
                    prec_flag = validate_precision(value)
                    cons_flag = 1
                    val_flag = 1
                    integ_flag =1 
                else:
                    prec_flag = 1
                    cons_flag = validate_consistency(value)
                    val_flag = validate_validity(value)
                    if col in columnas_integridad:
                        integ_flag = validate_integrity(value)
                    else:
                        integ_flag = 1
                
                if (prec_flag == 0 or cons_flag == 0 or val_flag == 0 or integ_flag == 0) and col not in columnas_unicidad:
                    output_data.append({
                        #'llave': f'{sheet_name}_{col}',
                        'fuente de información': sheet_name,
                        'Nomina' :nomina,
                        'campo': col,
                        'valor': value,
                        'bandera precisión': prec_flag,
                        'bandera consistencia': cons_flag,
                        'bandera integridad': integ_flag,
                        'bandera validez': val_flag
                    })
        
        output_df = pd.DataFrame(output_data)
        print(f'Generación de archivo de salida..')
        output_df.to_csv( f'Resultados_Perfilamiento/Perfilamiento_det_{sheet_name}.csv', index=False, encoding='utf-8-sig', sep='\t',quoting=csv.QUOTE_NONNUMERIC)
        #output_df.to_excel( f'Resultados_Perfilamiento/Perfilamiento_det_{sheet_name}.xlsx', index=False)
        print(f'ok!')


Inicio del proceso, espere unos minutos..


Pestaña analizada: DWH - Success Factor
Generación de archivo de salida..
ok!
Pestaña analizada: Profesor Escuela Top
Generación de archivo de salida..
ok!
Pestaña analizada: Profesor Facultad
Generación de archivo de salida..
ok!
Pestaña analizada: Profesor Grado Académico
Generación de archivo de salida..
ok!
